In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("init_load_flag", "0")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### Customers Dimentions using SCD Type 1


### Data Reading

In [0]:
df = spark.sql("select * from databricks035_catalog.silver.customers")

In [0]:
df.limit(10).display()

customer_id,email,city,state,domains,full_name
C01991,craigbarajas@williamson.com,Sylviafort,MS,williamson.com,Sonia Matthews
C01992,pcross@hotmail.com,New Anthonybury,MT,hotmail.com,Andrew Turner
C01993,careyjohn@ortiz.com,Port Larrymouth,TN,ortiz.com,Daniel Flynn
C01994,barkertaylor@hampton.info,North Ericshire,NJ,hampton.info,Nicole Griffin
C01995,millerjodi@hotmail.com,Wolffort,FL,hotmail.com,Vincent Long
C01996,alan72@salas.com,Villarrealtown,WV,salas.com,David Warren
C01997,raydana@sanders.com,New Gabriel,NM,sanders.com,Amber Lawson
C01998,beckyjones@hotmail.com,Davidland,NH,hotmail.com,Ivan Jenkins
C01999,brandondiaz@mcdowell.biz,Stephaniechester,TX,mcdowell.biz,Kathleen Hodges
C02000,mcdowellkatie@yahoo.com,West Dianechester,DE,yahoo.com,Julie Smith


#### Removing Duplicates from primary key (subset=['customer_id'])

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

In [0]:
df.limit(10).display()

customer_id,email,city,state,domains,full_name
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy


### Dividing old records with new ones

In [0]:
if init_load_flag == 0:
    df_old=spark.sql('''select DimCustomerKey, customer_id, create_date, update_date
                     from databricks035_catalog.gold.DimCustomers''')
    
else:
    df_old=spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date -- psuedo column creation and we just need names, by this we created a dataframe df_old with no values just column names why so that we can get joining key 
                     from databricks035_catalog.silver.customers where 1=0''')

In [0]:
df_old.display()

DimCustomerKey,customer_id,create_date,update_date
1,C00049,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
2,C00126,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
3,C00169,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
4,C00394,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
5,C00509,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
6,C00929,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
7,C01462,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
8,C01720,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
9,C01815,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
10,C00006,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z


#### Renaming Columns of df_old

In [0]:
df_old =  df_old.withColumnRenamed("DimCustomerKey", "old_DimCustoemrKey")\
                    .withColumnRenamed("customer_id", "old_customer_id")\
                    .withColumnRenamed("create_date", "old_create_date")\
                    .withColumnRenamed("update_date", "old_update_date") 

#### Applying Join with old records

In [0]:
df_join = df.join(df_old,df['customer_id'] == df_old['old_customer_id'], 'left')
df_join.display() # Since this will be our initial load we will have all values for DimCustomerKey,customer_id, create_date,update_date as NULL

customer_id,email,city,state,domains,full_name,old_DimCustoemrKey,old_customer_id,old_create_date,old_update_date
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,1,C00049,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2,C00126,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,3,C00169,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,4,C00394,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,5,C00509,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,6,C00929,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,7,C01462,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,8,C01720,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,9,C01815,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,10,C00006,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z


####Seperating New vs Old Records

In [0]:
df_new = df_join.filter(df_join['old_DimCustoemrKey'].isNull())
df_new.limit(10).display()

customer_id,email,city,state,domains,full_name,old_DimCustoemrKey,old_customer_id,old_create_date,old_update_date


In [0]:
df_old = df_join.filter(df_join['old_DimCustoemrKey'].isNotNull()) #df_old means that it is already present in the gold layer and we do not need to touch anything in it just update them in update_date which will be current time
df_old.limit(10).display()

customer_id,email,city,state,domains,full_name,old_DimCustoemrKey,old_customer_id,old_create_date,old_update_date
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,1,C00049,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2,C00126,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,3,C00169,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,4,C00394,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,5,C00509,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,6,C00929,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,7,C01462,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,8,C01720,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,9,C01815,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,10,C00006,2026-08-24T20:17:25.06Z,2026-08-24T20:17:25.06Z


#### Preparing df_old 
##### by preparing we mean make it beautiful accoring to destination table, and we do not need old_ columns and create  create and update columns as well 

In [0]:
# Dropping the columns which are not required and adding an update date column to the dataframe
df_old = df_old.drop('old_customer_id','old_update_date')

# Renaming old_DimCustomerKey to DimCustomerKey
df_old = df_old.withColumnRenamed("old_DimCustoemrKey", "DimCustomerKey")

# Renaming old_create_date column to create_date
df_old = df_old.withColumnRenamed("old_create_date", "create_date")
df_old = df_old.withColumn("create_date",to_timestamp(col("create_date")))

# Recreating update_date column with current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())

df_old.display()

customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,1,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,3,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,4,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,5,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,6,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,7,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,8,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,9,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,10,2026-08-24T20:17:25.06Z,2026-08-24T20:22:29.062Z


#### Preparing df_new


In [0]:
df_new.display()

customer_id,email,city,state,domains,full_name,old_DimCustoemrKey,old_customer_id,old_create_date,old_update_date


In [0]:
# Dropping the columns which are not required and adding an update date column to the dataframe
df_new = df_new.drop('old_DimCustoemrKey','old_customer_id','old_update_date','old_create_date')

# Recreating update_date column with current timestamp
df_new = df_new.withColumn("create_date", current_timestamp())
df_new = df_new.withColumn("update_date", current_timestamp())


df_new.limit(10).display()

customer_id,email,city,state,domains,full_name,create_date,update_date


#### Surrogate Key - From 1  

In [0]:
df_new = df_new.withColumn("DimCustomerKey", monotonically_increasing_id()+lit(1))
df_new.limit(5).display()

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey


#### Adding Max Surrogate key

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0

else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks035_catalog.gold.DimCustomers")
    # Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key)+col("DimCustomerKey"))

#### Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,1
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,2
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,3
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,4
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,5
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,6
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,7
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,8
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,9
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,2026-08-24T20:17:25.06Z,2026-08-24T20:24:30.671Z,10


### Applying UPSERT (UPDATE+INSERT) Condition a.k.a SCD Type - 1

In [0]:
# alternative of widgit flag
if spark.catalog.tableExists("databricks035_catalog.gold.DimCustomers"):
    print("Table exists")
else:
    print("Table is not present")

Table exists


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks035_catalog.gold.DimCustomers"): # appling upsert logic 
    
    dlt_obj = DeltaTable.forPath(spark,"abfss://gold@storageaccount03five.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("target").merge(df_final.alias("source"),"target.DimCustomerKey = source.DimCustomerKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else: 
    df_final.write.mode("overwrite")\
    .format("delta")\
    .option("path","abfss://gold@storageaccount03five.dfs.core.windows.net/DimCustomers")\
    .saveAsTable("databricks035_catalog.gold.DimCustomers")
    

In [0]:
%sql
SELECT * FROM databricks035_catalog.gold.dimcustomers

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,1
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,2
C00169,michaelfox@hotmail.com,Michaelburgh,DC,hotmail.com,Holly Ward,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,3
C00394,thomasmontgomery@gonzalez-ross.com,North Denisemouth,NJ,gonzalez-ross.com,Anthony Kelly,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,4
C00509,dylanmclean@franco-little.org,Pachecotown,TX,franco-little.org,Andre Murray,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,5
C00929,buchananjerry@evans-allen.biz,Wellsshire,NC,evans-allen.biz,Timothy Atkinson,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,6
C01462,sflynn@yahoo.com,Justinstad,MD,yahoo.com,Melody Daniels,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,7
C01720,carolmartin@yahoo.com,North Brianview,IN,yahoo.com,Cindy Gonzalez,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,8
C01815,aperez@gmail.com,Port Michaelstad,ID,gmail.com,Laura Kramer,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,9
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,2026-08-24T20:17:25.06Z,2026-08-24T20:25:20.114Z,10
